# XAI — Spatial k-means clustering of LRP composite maps

Segment the ensemble-mean LRP composite maps into spatial "regions of similar relevance" using k-means **on grid cells**.

### What is being clustered

- **Not** across samples/events. LRP is first composited across all 45 models (N_SPLITS × N_SEEDS) and across the relevant samples (TP or TN), giving a **single 2-D map per category**.
- K-means is then applied **to the pixels of that single map** (features = LRP value at each grid cell).
- The result is a spatial segmentation of the composite into clusters.

Two cases are built for each of TP / TN:
1. **signed LRP** (positive + negative relevance retained)
2. **positive-only LRP** (negative relevance zeroed, matching the project’s existing figures)

### Figure layout (per case)

```
            TP                TN
 row 1 : (a) frequency     (e) frequency
 row 2 : (b) composite     (f) composite
 row 3 : (c) cluster 1     (g) cluster 1
 row 4 : (d) cluster 2     (h) cluster 2
   …    : more rows if n_clusters > 2
```

Panel letters are assigned in reading order based on the total number of panels.

### Methodology choices

- **Normalisation** before clustering uses the project’s established **max-abs** convention
  (`normalize_map(..., method="maxabs")`). The signed map therefore lives in `[-1, 1]` and the
  positive-only map in `[0, 1]`, so threshold values have a consistent meaning across figures.
- **Clustering features.** For the signed case we cluster on the **raw signed LRP value**, so clusters
  naturally separate strongly-positive, near-zero, and strongly-negative regions. For the positive-only
  case we cluster on the positive LRP value, separating high-relevance regions from low-relevance ones.
- **Threshold-frequency** (row 1). For a signed composite, `|LRP_norm| > 0.3` is the cleanest single
  panel because it puts positive and negative exceedance on the same footing and is directly comparable
  to the positive-only case. The frequency is computed across the 45 per-model maps (each normalised
  by its own max-abs) **before** they are averaged into the composite — this reflects how often each
  grid cell is a strong-relevance cell for an individual model, rather than just thresholding the
  ensemble mean. See `src.xai.k_means.threshold_frequency_map`.
- **Number of clusters.** Defaults to `n_clusters = 2`. A tiny inertia / silhouette diagnostic is shown
  below so this can be adjusted if the LRP structure warrants more clusters (especially in the signed
  case where positive, near-zero, and negative regions suggest k = 3).

### Prerequisites

- Run `cnn_predict.ipynb` first (or at least the load-loop cells) so models and LRP attributions are on
  disk. This notebook reuses the same loading pattern.

## Imports & config

In [ ]:
import sys
from pathlib import Path
from string import ascii_lowercase

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cmocean

from sklearn.metrics import precision_recall_curve, silhouette_score

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from configs import paths
from src.cnn.splits import load_tvt_split
from src.cnn.train  import load_model, predict_splits
from src.xai.k_means import (
    normalize_map,
    prepare_spatial_features,
    run_kmeans_spatial,
    reshape_clusters,
    compute_cluster_means,
    sort_clusters_by_mean,
    threshold_frequency_map,
)

# Must match training
N_SPLITS  = 9
N_SEEDS   = 5
BASE_SEED = 42

## 1. Load LRP, CNN predictions, and true labels (training data only)

Reuses the same loading convention as `cnn_predict.ipynb`. For every split × seed we store:
- `lrp_flat`  : (n_lrp, nx, ny) LRP attribution maps
- `y_t`       : (n_tr,)         true training labels
- `y_pred`    : (n_tr,)         thresholded CNN predictions

In [ ]:
per_model = []   # one dict per split×seed
lat_shared, lon_shared = None, None

for split_idx in range(N_SPLITS):
    split_path = paths.tvt_split_path(split_idx)
    split = load_tvt_split(split_path)
    x_tr = split['sst_tr'][:, :, :, np.newaxis]
    x_va = split['sst_va'][:, :, :, np.newaxis]
    x_te = split['sst_te'][:, :, :, np.newaxis]
    y_tr = split['slow_tr']

    for run_idx in range(N_SEEDS):
        lrp_path = paths.attribution_path(split_idx, run_idx)
        if not lrp_path.exists():
            continue

        model = load_model(paths.MODELS_DIR, split_idx, run_idx)
        scores = predict_splits(model, x_tr, x_va, x_te)['train']

        prec, rec, thr = precision_recall_curve(y_tr, scores)
        thr_best = thr[np.argmin(np.abs(prec[:-1] - rec[:-1]))] if thr.size > 0 else 0.5
        y_pred = (scores >= thr_best).astype(int)

        with xr.open_dataset(lrp_path) as lrp_ds:
            lrp_raw = lrp_ds['lrp_attributions'].values          # (nch, nt, nx, ny, 1)
            if lat_shared is None:
                lat_shared = lrp_ds['lat'].values
                lon_shared = lrp_ds['lon'].values
        lrp_flat = lrp_raw.reshape(-1, lrp_raw.shape[2], lrp_raw.shape[3])

        n_lrp = lrp_flat.shape[0]
        per_model.append({
            'lrp_flat': lrp_flat,
            'y_t'     : y_tr[:n_lrp],
            'y_pred'  : y_pred[:n_lrp],
        })

print(f'Loaded LRP for {len(per_model)} split×seed models')

## 2. Build ensemble-mean composites for TP / TN

For each model we average LRP maps over the samples matching the category
(TP or TN), then average the resulting per-model maps across all 45 models.
We also keep the **stack of per-model means** because the threshold-frequency
panel counts how often each grid cell is a strong-relevance cell across the
ensemble (before the mean).

In [ ]:
def build_composites(per_model, category, positive_only=False):
    '''Return (composite_map, stack_of_per_model_maps).

    category : 'TP' or 'TN'
    positive_only : if True, negative LRP values are zeroed (matches
                    the positive-only figure style in cnn_predict.ipynb).
    '''
    per_map = []
    for m in per_model:
        y_t, y_pred = m['y_t'], m['y_pred']
        if category == 'TP':
            mask = (y_pred == 1) & (y_t == 1)
        elif category == 'TN':
            mask = (y_pred == 0) & (y_t == 0)
        else:
            raise ValueError(category)
        if mask.sum() == 0:
            continue
        sel = m['lrp_flat'][mask]
        if positive_only:
            sel = np.where(sel > 0, sel, 0.0)
        per_map.append(sel.mean(axis=0))
    stack = np.stack(per_map, axis=0)         # (n_models, nx, ny)
    composite = stack.mean(axis=0)            # (nx, ny)
    return composite, stack


tp_signed, tp_signed_stack = build_composites(per_model, 'TP', positive_only=False)
tn_signed, tn_signed_stack = build_composites(per_model, 'TN', positive_only=False)
tp_pos,    tp_pos_stack    = build_composites(per_model, 'TP', positive_only=True)
tn_pos,    tn_pos_stack    = build_composites(per_model, 'TN', positive_only=True)

print('signed composites :', tp_signed.shape, tn_signed.shape)
print('positive-only     :', tp_pos.shape,    tn_pos.shape)

## 3. Quick diagnostic: inertia / silhouette vs. k

Justify the default `n_clusters` by scanning a small range of `k` for each
composite map. This is a sanity-check only — the notebook does not rely on
it automatically picking `k`.

In [ ]:
from sklearn.cluster import KMeans

def kmeans_diagnostic(lrp_map, k_range=(2, 3, 4, 5)):
    feats, _ = prepare_spatial_features(normalize_map(lrp_map, 'maxabs'))
    out = {}
    for k in k_range:
        km = KMeans(n_clusters=k, random_state=0, n_init=10).fit(feats)
        # silhouette on a sub-sample to keep it fast
        n = feats.shape[0]
        idx = np.random.default_rng(0).choice(n, size=min(5000, n), replace=False)
        sil = silhouette_score(feats[idx], km.labels_[idx]) if k > 1 else np.nan
        out[k] = {'inertia': float(km.inertia_), 'silhouette': float(sil)}
    return out

for label, m in [('TP signed', tp_signed), ('TN signed', tn_signed),
                  ('TP pos',    tp_pos),    ('TN pos',    tn_pos)]:
    print(f'\n{label}:')
    for k, s in kmeans_diagnostic(m).items():
        print(f'  k={k}: inertia={s["inertia"]:9.2f}   silhouette={s["silhouette"]:.3f}')

## 4. Cluster the four composite maps

`n_clusters` is set globally below. If the signed diagnostic above prefers
`k = 3` (one positive region, one near-zero, one negative), set
`N_CLUSTERS_SIGNED = 3` and re-run.  The positive-only case is well-served
by `k = 2` (high-relevance vs. low-relevance).

In [ ]:
N_CLUSTERS_SIGNED = 2      # try 3 if signed LRP shows clear +/0/− structure
N_CLUSTERS_POS    = 2


def cluster_composite(lrp_map, n_clusters, seed=0):
    '''Normalise, cluster, reshape, and sort the clusters by mean LRP.'''
    norm = normalize_map(lrp_map, method='maxabs')
    feats, valid = prepare_spatial_features(norm)
    labels = run_kmeans_spatial(feats, n_clusters=n_clusters, random_state=seed)
    cmap = reshape_clusters(labels, valid)
    cmap = sort_clusters_by_mean(cmap, norm, descending=True)
    return norm, cmap


tp_signed_norm, tp_signed_clusters = cluster_composite(tp_signed, N_CLUSTERS_SIGNED)
tn_signed_norm, tn_signed_clusters = cluster_composite(tn_signed, N_CLUSTERS_SIGNED)
tp_pos_norm,    tp_pos_clusters    = cluster_composite(tp_pos,    N_CLUSTERS_POS)
tn_pos_norm,    tn_pos_clusters    = cluster_composite(tn_pos,    N_CLUSTERS_POS)

print('TP signed cluster means:', compute_cluster_means(tp_signed_norm, tp_signed_clusters))
print('TN signed cluster means:', compute_cluster_means(tn_signed_norm, tn_signed_clusters))
print('TP pos    cluster means:', compute_cluster_means(tp_pos_norm,    tp_pos_clusters))
print('TN pos    cluster means:', compute_cluster_means(tn_pos_norm,    tn_pos_clusters))

## 5. Threshold-frequency maps

Frequency across the 45 per-model maps of `|LRP_norm| > 0.3` for the signed
case, and of `LRP_norm > 0.3` for the positive-only case.  Each per-model map
is normalised by its own max-abs so `0.3` means the same thing across models.

In [ ]:
THRESH = 0.3

tp_signed_freq = threshold_frequency_map(tp_signed_stack, threshold=THRESH, mode='absolute')
tn_signed_freq = threshold_frequency_map(tn_signed_stack, threshold=THRESH, mode='absolute')
tp_pos_freq    = threshold_frequency_map(tp_pos_stack,    threshold=THRESH, mode='positive')
tn_pos_freq    = threshold_frequency_map(tn_pos_stack,    threshold=THRESH, mode='positive')

## 6. Plotting

Plot helper that produces the two-column, multi-row figure described in the
intro. It matches the projection, coastlines, and colormap conventions used
in the SST/LRP composite cells of `cnn_predict.ipynb`:
- signed LRP → `cmocean.cm.curl` on `[−1, 1]`
- positive LRP → `magma` on `[0, 1]`
- frequency → `viridis` on `[0, 1]`

Cluster rows show the **masked LRP map** for that cluster (values inside
the cluster, NaN elsewhere) so the visual scale matches the composite panel
directly above.

In [ ]:
def plot_kmeans_figure(
    tp_freq, tn_freq,
    tp_norm, tn_norm,
    tp_clusters, tn_clusters,
    lat, lon,
    signed=True,
    title_suffix='',
):
    '''2-column, (2 + n_clusters)-row figure.  Columns = TP | TN.'''
    n_c = max(int(np.nanmax(tp_clusters)), int(np.nanmax(tn_clusters))) + 1
    n_rows = 2 + n_c
    proj  = ccrs.PlateCarree(central_longitude=180)
    lon2d, lat2d = np.meshgrid(lon, lat)

    fig, axes = plt.subplots(
        n_rows, 2, figsize=(14, 3.2 * n_rows),
        subplot_kw={'projection': proj},
    )

    # colormaps / limits
    if signed:
        comp_cmap = cmocean.cm.curl
        comp_vmin, comp_vmax = -1.0, 1.0
        comp_label = 'Normalised relevance (signed)'
    else:
        comp_cmap = 'magma'
        comp_vmin, comp_vmax = 0.0, 1.0
        comp_label = 'Normalised relevance (positive)'

    freq_cmap = 'viridis'

    # Flatten panel list in reading order (row by row, TP then TN)
    panel_specs = []
    panel_specs.append(('freq', axes[0, 0], tp_freq, 'TP'))
    panel_specs.append(('freq', axes[0, 1], tn_freq, 'TN'))
    panel_specs.append(('comp', axes[1, 0], tp_norm, 'TP'))
    panel_specs.append(('comp', axes[1, 1], tn_norm, 'TN'))
    for ci in range(n_c):
        tp_mask = np.where(tp_clusters == ci, tp_norm, np.nan)
        tn_mask = np.where(tn_clusters == ci, tn_norm, np.nan)
        panel_specs.append(('cluster', axes[2 + ci, 0], tp_mask, f'TP · cluster {ci}'))
        panel_specs.append(('cluster', axes[2 + ci, 1], tn_mask, f'TN · cluster {ci}'))

    letters = iter(ascii_lowercase)

    for kind, ax, data, tag in panel_specs:
        letter = next(letters)
        ax.set_global()
        ax.add_feature(cfeature.LAND, facecolor='lightgray', zorder=1)
        ax.add_feature(cfeature.COASTLINE, linewidth=0.4, zorder=2)
        ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.5)

        if kind == 'freq':
            im = ax.pcolormesh(lon2d, lat2d, data,
                               cmap=freq_cmap, vmin=0.0, vmax=1.0,
                               transform=ccrs.PlateCarree(), shading='auto', zorder=0)
            cb_label = f'P(|LRP_norm| > {THRESH})' if signed else f'P(LRP_norm > {THRESH})'
            ax.set_title(f'({letter}) {tag}: frequency |LRP| > {THRESH}' if signed
                         else f'({letter}) {tag}: frequency LRP > {THRESH}', fontsize=11)
        else:
            im = ax.pcolormesh(lon2d, lat2d, data,
                               cmap=comp_cmap, vmin=comp_vmin, vmax=comp_vmax,
                               transform=ccrs.PlateCarree(), shading='auto', zorder=0)
            cb_label = comp_label
            if kind == 'comp':
                ax.set_title(f'({letter}) {tag}: composite LRP', fontsize=11)
            else:
                ax.set_title(f'({letter}) {tag}', fontsize=11)

        plt.colorbar(im, ax=ax, orientation='horizontal', pad=0.04,
                     fraction=0.046, label=cb_label)

    fig.suptitle(
        f'Spatial k-means clustering of ensemble-mean LRP {title_suffix}\n'
        f'({N_SPLITS} splits × {N_SEEDS} seeds  →  {len(per_model)} per-model maps)',
        fontsize=13, y=1.01,
    )
    plt.tight_layout()
    return fig

### Case 1: signed LRP (positive + negative)

Clusters split the map into regions of similar **signed** normalised relevance.
With `k = 2`, this usually becomes "positive region" vs. "everything else".
Row 1 shows how often each grid cell exceeds `|LRP_norm| > 0.3` across the 45
models — a single, sign-agnostic panel that is directly comparable to the
positive-only case below.

In [ ]:
fig = plot_kmeans_figure(
    tp_signed_freq, tn_signed_freq,
    tp_signed_norm, tn_signed_norm,
    tp_signed_clusters, tn_signed_clusters,
    lat_shared, lon_shared,
    signed=True,
    title_suffix='(signed)',
)
plt.show()

### Case 2: positive-only LRP

Matches the positive-only figures in `cnn_predict.ipynb`.  Clusters separate
high-relevance regions from low-relevance background.  Row 1 shows
`LRP_norm > 0.3` (no absolute value needed, since negative relevance has been
zeroed).

In [ ]:
fig = plot_kmeans_figure(
    tp_pos_freq, tn_pos_freq,
    tp_pos_norm, tn_pos_norm,
    tp_pos_clusters, tn_pos_clusters,
    lat_shared, lon_shared,
    signed=False,
    title_suffix='(positive-only)',
)
plt.show()

## Summary

- **Clustering target.** Spatial k-means on a single 2-D composite map (pixels = samples, feature = LRP value). No clustering across events.
- **Normalisation.** Max-abs per composite, applied before clustering and for display. Matches the existing LRP plotting convention.
- **Threshold-frequency.** Computed across the 45 per-model composite maps before averaging, with each map self-normalised. Signed case uses `|LRP_norm| > 0.3`; positive-only case uses `LRP_norm > 0.3`. This is more informative than thresholding the ensemble mean because it captures how consistently each grid cell is a strong-relevance cell across models.
- **n_clusters.** Default `k = 2` for both cases; the diagnostic cell reports inertia and silhouette so this can be raised (e.g. `k = 3` for the signed case) if the data supports it.